# AgriInsight: Machine Learning Crop Yield Prediction Suite
### Supervised Regression Modeling, Benchmarking & Model Evaluation

This notebook documents the predictive machine learning workflow:
1. **Feature Ingestion & Column Transformation** (OneHotEncoding + Standard Scaling)
2. **Algorithm Benchmarking**:
   - Baseline: Ordinary Least Squares (Linear Regression)
   - Regularized: Ridge Regression (L2)
   - Ensemble: Random Forest Regressor
   - Boosting: Gradient Boosting Regressor
3. **Metrics Evaluation**: $R^2$ Score, RMSE, MAE, and MAPE
4. **5-Fold Cross-Validation**
5. **Feature Importance & Residual Error Diagnostics**
6. **Model Serialization and Test Inference**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Load Preprocessed Data and Train/Test Split

In [ ]:
clean_data_path = os.path.join('..', 'data', 'processed', 'agricultural_yield_cleaned.csv')
df = pd.read_csv(clean_data_path)

numeric_features = [
    'Annual_Rainfall_mm', 'Seasonal_Rainfall_mm', 'Temperature_Avg_C',
    'Humidity_Pct', 'Soil_pH', 'Soil_Nitrogen_N', 'Soil_Phosphorus_P',
    'Soil_Potassium_K', 'NPK_Total', 'N_to_P_Ratio', 'Fertilizer_Usage_Kg_Per_Ha',
    'Pesticide_Usage_Kg_Per_Ha', 'Irrigation_Coverage_Pct', 'Area_Hectares'
]
categorical_features = ['Crop', 'Season', 'Soil_Type', 'Zone']
target = 'Yield_Tons_Per_Ha'

X = df[numeric_features + categorical_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Train sample count: {len(X_train)}, Test sample count: {len(X_test)}")

## 2. Load Model Benchmark Results

In [ ]:
benchmarks_path = os.path.join('..', 'reports', 'model_benchmarks.csv')
benchmarks_df = pd.read_csv(benchmarks_path)
benchmarks_df

## 3. Load Winning Model and Evaluate Performance Diagnostics

In [ ]:
best_model = joblib.load(os.path.join('..', 'models', 'best_crop_yield_model.joblib'))
preprocessor = joblib.load(os.path.join('..', 'models', 'preprocessor.joblib'))

X_test_trans = preprocessor.transform(X_test)
y_pred = best_model.predict(X_test_trans)

print(f"Test R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"Test RMSE:     {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test MAE:      {mean_absolute_error(y_test, y_pred):.4f}")
print(f"Test MAPE:     {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}%")

## 4. Residuals & Error Distribution Plot

In [ ]:
residuals = y_test - y_pred
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(y_test, y_pred, alpha=0.4, color='#1f77b4')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_title('Actual vs Predicted Crop Yield (MT/Ha)', weight='bold')
ax1.set_xlabel('Actual Yield (MT/Ha)')
ax1.set_ylabel('Predicted Yield (MT/Ha)')

sns.histplot(residuals, kde=True, ax=ax2, color='#2ca02c', bins=30)
ax2.axvline(0, color='red', linestyle='--')
ax2.set_title('Prediction Error Residuals Distribution', weight='bold')
ax2.set_xlabel('Residuals (Actual - Predicted)')
plt.tight_layout()
plt.show()

## 5. Live Inference Demonstration using AgriYieldPredictor

In [ ]:
import sys
sys.path.append('..')
from src.predict import AgriYieldPredictor

predictor = AgriYieldPredictor(
    model_path=os.path.join('..', 'models', 'best_crop_yield_model.joblib'),
    preprocessor_path=os.path.join('..', 'models', 'preprocessor.joblib'),
    metadata_path=os.path.join('..', 'models', 'model_metadata.joblib')
)

sample_case = {
    'Crop': 'Rice',
    'Season': 'Kharif',
    'Soil_Type': 'Alluvial',
    'Zone': 'East',
    'Annual_Rainfall_mm': 1350.0,
    'Seasonal_Rainfall_mm': 980.0,
    'Temperature_Avg_C': 27.5,
    'Humidity_Pct': 82.0,
    'Soil_pH': 6.8,
    'Soil_Nitrogen_N': 115.0,
    'Soil_Phosphorus_P': 52.0,
    'Soil_Potassium_K': 48.0,
    'Fertilizer_Usage_Kg_Per_Ha': 210.0,
    'Pesticide_Usage_Kg_Per_Ha': 2.8,
    'Irrigation_Coverage_Pct': 75.0,
    'Area_Hectares': 2500.0
}
pred_output = predictor.predict(sample_case)
pd.DataFrame([pred_output])